# Figure 8 — Opinion trajectories and space–time communities

Run all cells from `REVIEWED_CODE`. The Section 4 reduced model and k-means use the existing adjacency matrices in `PAPER_EXAMPLES/example_3/network`, including **all 503 nodes**. Spatial modes **1 and 3** are used with **four clusters**. No adjacency matrices are reconstructed from trajectories.

`trajectories.npy` supplies plotting positions for panels (a) and (b) only. Its first 500 nodes are voters and its last three are parties. Panel (a) displays both; panel (b) displays voters with their labels from the full 503-node clustering. Panel (c) displays all network nodes, ordered by mode 1 at the second snapshot, without using trajectory values.

Only 40 sampled times are available for panel (a). The reduced basis dimension is configurable (default 4).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
from scipy.optimize import linear_sum_assignment
from reduced_spatiotemporal_clustering import compute_reduced_spatial_eigenvectors
from spatiotemporal_clustering import cluster_embedding, load_temporal_network

NETWORK_DIR = Path('PAPER_EXAMPLES/example_3/network')
TRAJECTORIES_PATH = Path('PAPER_EXAMPLES/example_3/trajectories.npy')
N_PARTIES = 3  # last three trajectories; remaining nodes are voters
SNAPSHOT_STRIDE = 10  # simulation steps: 0, 10, ..., 390
ALPHA = 0.01
BASIS_DIMENSION = 4  # includes constant; increase to check convergence
CUSTOM_BASES = None  # optional one (N_network, d_t) array per snapshot
SELECTED_EIGENVECTORS = [1, 3]  # 1-based
N_CLUSTERS = 4
RANDOM_STATE = 0
SAVE_FIGURE = False
OUTPUT_DIR = Path('figure_8_output')


## 1. Load the existing network and plotting trajectories

Adjacency files are sorted numerically by the shared loader. All nodes and supplied edge weights enter the reduced model. Trajectory rows must match the snapshot count and columns must match the network node ordering. Trajectories are used only for panels (a) and (b), including their presentation colors.

In [ ]:
adjacencies, snapshot_files = load_temporal_network(NETWORK_DIR)
M, N = len(adjacencies), adjacencies[0].shape[0]
raw = np.load(TRAJECTORIES_PATH, allow_pickle=False)
if raw.ndim == 3 and raw.shape[-1] == 1:
    raw = raw[..., 0]
if raw.ndim != 2 or not np.all(np.isfinite(raw)):
    raise ValueError('Expected finite trajectories of shape (snapshots, nodes, 1) or (snapshots, nodes)')
if raw.shape != (M, N):
    raise ValueError(f'Trajectory shape {raw.shape} must match network shape {(M, N)} and node ordering')
if not 0 < N_PARTIES < N or M < 2:
    raise ValueError('Need at least two snapshots, voters, and party trajectories')
voters, parties = raw[:, :-N_PARTIES], raw[:, -N_PARTIES:]
N_VOTERS = voters.shape[1]
simulation_steps = np.arange(M) * SNAPSHOT_STRIDE
snapshots = np.arange(1, M + 1)
print(f'Loaded {M} adjacency matrices with {N} nodes from {NETWORK_DIR.resolve()}')
print('First/last snapshot:', snapshot_files[0].name, '/', snapshot_files[-1].name)
print(f'Plotting {N_VOTERS} voters and {N_PARTIES} parties')


## 2. Reduced spatial spectrum and clustering

Use noncyclic exponential coupling with $\alpha=0.01$. Lift reduced eigenvectors to **all network nodes** before applying k-means to modes 1 and 3. No row normalization is applied. Labels are assigned to individual snapshot–node pairs, allowing communities to merge over time. Neither the eigensolver nor k-means receives trajectory values.

In [ ]:
result = compute_reduced_spatial_eigenvectors(
    adjacencies, alpha=ALPHA, cyclic=False,
    n_eigenvectors=max(3, max(SELECTED_EIGENVECTORS)),
    basis_dimension=BASIS_DIMENSION, bases=CUSTOM_BASES,
    random_state=RANDOM_STATE,
)
labels, embedding = cluster_embedding(
    result.eigenvectors, SELECTED_EIGENVECTORS, N_CLUSTERS,
    random_state=RANDOM_STATE, row_normalize=False,
)
print('Spatial eigenvalues:', result.eigenvalues)
print('Reduced spatial dimension:', result.spatial_operator.shape[0])
assert labels.shape == (M, N)
voter_labels = labels[:, :N_VOTERS]  # plotting subset only; clustering includes parties
assert np.all(np.isfinite(result.eigenvectors))


## 3. Match the four cluster colors to Figure 8


In [ ]:
if N_CLUSTERS != 4:
    raise ValueError('The Figure 8 palette requires N_CLUSTERS = 4')
late_label = int(np.argmax(np.bincount(voter_labels[-1], minlength=4)))
remaining = [k for k in range(4) if k != late_label]
initial_centers = np.sort(parties[0])
if len(initial_centers) != 3:
    raise ValueError('Automatic Figure 8 colors assume three parties')
initial_groups = np.argmin(abs(voters[0, :, None] - initial_centers[None, :]), axis=1)
counts = np.array([[np.count_nonzero((voter_labels[0] == k) & (initial_groups == g))
                    for k in remaining] for g in range(3)])
rows, cols = linear_sum_assignment(-counts)
initial_labels = np.empty(3, dtype=int)
initial_labels[rows] = np.asarray(remaining)[cols]
COLOR_ORDER = None  # optional [low_opinion_label, middle_label, high_label, late_label]
color_order = list(initial_labels) + [late_label] if COLOR_ORDER is None else list(COLOR_ORDER)
if sorted(color_order) != list(range(4)):
    raise ValueError('COLOR_ORDER must be a permutation of [0, 1, 2, 3]')
label_to_color = np.empty(4, dtype=int)
label_to_color[color_order] = np.arange(4)
colored_labels = label_to_color[labels]
cluster_cmap = ListedColormap(['#ff0000', '#ff9900', '#00c8ff', '#000080'])
cluster_norm = BoundaryNorm(np.arange(5) - 0.5, 4)
print('Labels mapped to red, orange, light blue, dark blue:', color_order)


## 4. Plot Figure 8 and save results

Panel (a): sampled voter and party trajectories. Panel (b): voter dots colored by their labels from clustering the supplied network. Panel (c): computed modes 1 and 3 for all 503 network nodes, ordered by mode 1 at the **second snapshot**.

In [ ]:
fig = plt.figure(figsize=(14, 8), layout='constrained')
grid = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])
ax_a = fig.add_subplot(grid[0, 0])
ax_b = fig.add_subplot(grid[0, 1])
ax_c = [fig.add_subplot(grid[1, j]) for j in range(2)]

ax_a.plot(simulation_steps, voters, color='#1f77b4', linewidth=0.35, alpha=0.18)
ax_a.plot(simulation_steps, parties, color='red', linewidth=1.5)
for step in simulation_steps:
    ax_a.axvline(step, color='0.4', linestyle='--', linewidth=0.4, alpha=0.5)
ax_a.set(title="Hegselmann–Krause model: voters’ and parties’ trajectories",
         xlabel='Time (simulation step)', ylabel='Opinion', ylim=(0, 1),
         xlim=(simulation_steps[0], simulation_steps[-1]))
ax_a.text(-0.10, 1.03, 'a)', transform=ax_a.transAxes, fontsize=14)

ax_b.scatter(np.repeat(snapshots, N_VOTERS), voters.ravel(), c=colored_labels[:, :N_VOTERS].ravel(),
             cmap=cluster_cmap, norm=cluster_norm, s=5, linewidths=0, rasterized=True)
ax_b.set(title='Communities of voters in the temporal network', xlabel='Snapshots',
         ylabel='Opinion', ylim=(0, 1.02), xlim=(0, M + 1))
ax_b.set_xticks([1] + list(range(5, M + 1, 5)))
ax_b.text(-0.10, 1.03, 'b)', transform=ax_b.transAxes, fontsize=14)

modes = result.eigenvectors[:, :, [0, 2]].copy()
node_order = np.argsort(modes[1, :, 0], kind='stable')
heat_norm = Normalize(vmin=-1.5, vmax=3)
for j, ax in enumerate(ax_c):
    im = ax.imshow(modes[:, node_order, j].T, cmap='coolwarm', norm=heat_norm,
                   aspect='auto', interpolation='nearest', extent=(0.5, M+0.5, N+0.5, 0.5))
    ax.set(title=f'Spatial eigenvector {[1, 3][j]}', xlabel='Snapshots', ylabel='Nodes')
    ax.set_xticks(np.arange(1, M+1, 2))
ax_c[0].text(-0.10, 1.03, 'c)', transform=ax_c[0].transAxes, fontsize=14)
fig.colorbar(im, ax=ax_c, shrink=0.95, label='Computed observable')

if SAVE_FIGURE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / 'figure_8.png', dpi=220)
    fig.savefig(OUTPUT_DIR / 'figure_8.pdf')
    np.savez_compressed(OUTPUT_DIR / 'figure_8_clustering.npz',
        labels=labels, color_labels=colored_labels, color_order=color_order,
        eigenvalues=result.eigenvalues, eigenvectors=result.eigenvectors,
        embedding=embedding, node_order=node_order, simulation_steps=simulation_steps,
        network_files=np.array([str(p) for p in snapshot_files]),
        alpha=ALPHA, basis_dimensions=[w.shape[1] for w in result.bases],
        selected_eigenvectors=SELECTED_EIGENVECTORS, random_state=RANDOM_STATE)
    print('Saved figure and clustering arrays to', OUTPUT_DIR.resolve())
plt.show()
